# M66b — why did Vehicle feature distillation add almost no gain?

Revision: **M66B-ZERO-UPDATE-KD-GRADIENTS-2026-10-06-r2** (2026-10-06).

**r2 correction:** checkpoint class keys `0/1` and JSON keys `"0"/"1"` are compared consistently; all other lineage values and weight hashes remain exact. The default output is now `gradient_r2`. Keep the original `gradient_r1` results untouched and rerun sections 1–3 from this updated notebook; matching private runtime, checkout and local data can be reused.

This is a **train-only, zero-update diagnostic**, not another training run. Compare original A2, the completed M66 no-KD control, and the completed M66 KD endpoint on the same first 64 training images. Measure the actual weighted GT, preservation and KD gradients: strength, conflict, coverage, and which head layers they reach.

Use a Colab **GPU** runtime (A100 recommended); CPU alone cannot execute native MonoDETR attention. No iPhone is needed. Run the **three sections below in order**. After a reset, rerun sections 1–3; matching completed endpoint reports are verified and reused. If the GPU/runtime/input identity changes, preserve existing reports and choose a new `RUN_ID`.

Required on Drive: the unchanged M66 manifest, smoke, paired gate, both endpoint weights/receipts/training summaries, original A2 weights and R0 weights. `m66_results.zip` alone contains no weights. This notebook restores a new CUDA13 private runtime; it does not reconstruct M62/M63 or mutate historical manifests.

**Deliverable:** `m66b_gradient_results.zip`. Stop for review. No optimizer steps, full validation, loss/mask changes, coefficient grid, teacher replacement, model promotion, or deployment are authorized. Original A2 remains selected.


## 1. Mount Drive, update main, and check all inputs

All imports, paths, helpers and revision checks are defined here. Edit only `RUN_ID` or genuinely relocated input paths; do not substitute different checkpoints.


In [ ]:
# M66B-ZERO-UPDATE-KD-GRADIENTS-2026-10-06-r2 — endpoint JSON class-key correction; preserve gradient_r1
from pathlib import Path
from collections import deque
import ast, hashlib, json, os, runpy, shlex, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

REVISION = 'M66B-ZERO-UPDATE-KD-GRADIENTS-2026-10-06-r2'
RUN_ID = 'gradient_r2'  # New checker identity; do not overwrite gradient_r1 reports.
MOBILE_REPO = Path('/content/mobile_adas3d')
A2_REPO = Path('/content/MonoDETR_M66B_A2')
MANIFEST = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m66_r0_a2_feature/m66_r0_a2_vehicle_feature_r1/m66_manifest.json')
OUTPUT = MANIFEST.parent/'diagnostics_m66b'/RUN_ID
DATASET = Path('/content/kitti_m66b')
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
PYTHON = Path('/content/m66b_cuda130_venv/bin/python')
CUDA = Path('/usr/local/cuda-13.0')
if not (CUDA/'bin/nvcc').is_file(): CUDA = Path('/usr/local/cuda')
RUNTIME_RECEIPT = OUTPUT/'m66b_runtime_receipt.json'
READY = False
ENV = os.environ.copy()
OUTPUT.mkdir(parents=True,exist_ok=True)
print(REVISION, '\nOutput:', OUTPUT, flush=True)

def run_logged(command, name, cwd=None):
    command = [str(x) for x in command]
    log_path = OUTPUT/'colab_logs'/f'{name}.log'
    log_path.parent.mkdir(parents=True,exist_ok=True)
    tail = deque(maxlen=60)
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True)
    with log_path.open('a') as log:
        log.write('\nCOMMAND: '+shlex.join(command)+'\n'); log.flush()
        with subprocess.Popen(command,cwd=str(cwd) if cwd else None,env=ENV,
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
            status = process.wait()
        log.write(f'\nEXIT CODE: {status}\n'); log.flush()
    if status: raise RuntimeError(f'Exit {status}; full log={log_path}\n'+'\n'.join(tail))

def checkout(path, url, commit):
    if '[' in url or ']' in url or not url.startswith('https://github.com/'):
        raise ValueError('Use a plain public GitHub URL, not Markdown')
    if not path.exists():
        run_logged(['git','clone','--no-checkout',url,path],f'clone_{path.name}')
        run_logged(['git','checkout','--detach',commit],f'pin_{path.name}',path)
    if not (path/'.git').is_dir(): raise RuntimeError(f'Preserve {path}; choose a fresh checkout path')
    head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=path,text=True).strip()
    origin = subprocess.check_output(['git','remote','get-url','origin'],cwd=path,text=True).strip()
    if head != commit or origin != url:
        raise RuntimeError(f'Unexpected checkout identity: {path}; do not reset it')

def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()

if not MOBILE_REPO.exists():
    run_logged(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone_mobile')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve other branch work')
run_logged(['git','pull','--ff-only'],'update_mobile',MOBILE_REPO)
SCRIPT = MOBILE_REPO/'scripts/probe_m66_kd_gradients.py'
if not SCRIPT.is_file(): raise FileNotFoundError('M66b is not published yet; reload the latest main notebook.')
revisions = [ast.literal_eval(node.value) for node in ast.parse(SCRIPT.read_text()).body
             if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='REVISION' for t in node.targets)]
if revisions != [REVISION]: raise RuntimeError('Notebook/script revisions differ; reload the latest notebook')
if not MANIFEST.is_file(): raise FileNotFoundError(MANIFEST)
m = json.loads(MANIFEST.read_text())
saved = m.get('manifest_sha256')
actual = hashlib.sha256(json.dumps({k:v for k,v in m.items() if k!='manifest_sha256'},sort_keys=True,allow_nan=False).encode()).hexdigest()
if actual != saved or saved != '68c2247a9ab28603e10f7f9748fc17a43e5bdefeedc57eebe87e2c62f430d968':
    raise RuntimeError('Use the unchanged reviewed M66 manifest; do not prepare a new M66 run')
required = [Path(m['models']['a2']['checkpoint']),Path(m['r0']['checkpoint']),
            MANIFEST.parent/'m66_paired_gate.json',MANIFEST.parent/'m66_training_smoke.json']
for role in ('control','kd'):
    required += [MANIFEST.parent/role/name for name in ('checkpoint_epoch_1.pth','checkpoint_epoch_1.json','training_summary.json')]
for path in required:
    if not path.is_file(): raise FileNotFoundError(f'Required M66 input missing: {path}; results ZIP has no weights')
print('Section 1 passed. All input files exist; exact byte identities are checked before GPU model loading.')


## 2. Restore the working CUDA13 runtime, source, native attention and data

Uses the same proven M64–M66 helpers, a fresh model checkout and isolated interpreter. Colab's global Torch packages are not changed. This section runs CPU regression tests, not student training. Source, weights, input receipts and environment are checked again by each probe process.


In [ ]:
READY = False
if 'SCRIPT' not in globals() or not SCRIPT.is_file(): raise RuntimeError('Run section 1 first')
helpers = runpy.run_path(str(MOBILE_REPO/'scripts/setup_m64_runtime.py'),run_name='shared_cuda13_helpers')
ENV = helpers['runtime_env'](CUDA,PYTHON.parent)
run_logged([sys.executable,'-u',MOBILE_REPO/'scripts/setup_m64_runtime.py',
            '--venv',PYTHON.parent.parent,'--cuda',CUDA,'--receipt',RUNTIME_RECEIPT],
           'm66b_runtime',MOBILE_REPO)
checkout(A2_REPO,'https://github.com/ZrrSkywalker/MonoDETR.git','6994b9f512400b258c6edb75f77423beb9c126f2')
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_mobilenetv4.py'):
    run_logged([PYTHON,MOBILE_REPO/'scripts'/patch,'--monodetr-repo',A2_REPO],patch.removesuffix('.py'),MOBILE_REPO)
run_logged([PYTHON,MOBILE_REPO/'scripts/patch_m64_inference_sources.py','--repo',A2_REPO,'--role','a2'],'m66b_patch_a2',MOBILE_REPO)
run_logged([PYTHON,'-X','faulthandler','-u',MOBILE_REPO/'scripts/build_m64_attention.py',
            '--repo',A2_REPO,'--role','a2'],'m66b_build_a2',MOBILE_REPO)
for test in ('test_m66_feature_pilot.py','test_m66b_gradients.py'):
    run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p',test,'-v'],test.removesuffix('.py'),MOBILE_REPO)
command = [PYTHON,MOBILE_REPO/'scripts/restore_m63_data.py','--dataset',DATASET,'--splits',SPLITS]
for candidate in ('/content/kitti_m66','/content/kitti_m65c','/content/kitti_m65','/content/kitti_m64',
                  '/content/kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti'):
    command += ['--candidate',candidate]
run_logged(command,'m66b_restore_data',MOBILE_REPO)
READY = True
print('Section 2 passed. Run section 3: only gradients, no optimizer or full validation.')


## 3. Measure the three endpoints, compare, bundle and stop

Each endpoint runs in a fresh subprocess on identical seeded training views. Completed matching reports are reusable; incomplete or changed identities are not overwritten. The bundle is also generated after a probe failure so its logs can be reviewed. Undefined cosines/ratios are `null`, not a claimed zero-strength conclusion.


In [ ]:
if 'READY' not in globals() or not READY: raise RuntimeError('Run sections 1 and 2 in order first')
try:
    for role in ('original','control','kd'):
        report_path = OUTPUT/f'{role}_gradients.json'
        run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,'probe',
                    '--manifest',MANIFEST,'--repo',A2_REPO,'--dataset-root',DATASET,
                    '--runtime-receipt',RUNTIME_RECEIPT,'--role',role,'--report',report_path],
                   f'm66b_{role}_gradients',MOBILE_REPO)
        report = json.loads(report_path.read_text())
        assert report['complete'] and report['optimizer_steps']==0 and all(report['checks'].values())
        assert not report['training_authorized'] and not report['checkpoint_promotion_authorized']
    run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,'summarize','--output',OUTPUT],
               'm66b_comparison',MOBILE_REPO)
    comparison = json.loads((OUTPUT/'m66b_gradient_comparison.json').read_text())
    assert comparison['paired_augmented_inputs_identical'] and comparison['optimizer_steps']==0
    print(json.dumps(comparison['roles'],indent=2))
finally:
    run_logged([PYTHON,'-u',SCRIPT,'bundle','--output',OUTPUT,'--manifest',MANIFEST],
               'm66b_bundle',MOBILE_REPO)
    print('Return:',OUTPUT/'m66b_gradient_results.zip')
print('STOP FOR REVIEW. Original A2 remains selected; no automatic training, coefficient changes or deployment.')
